<a href="https://colab.research.google.com/github/CaptainMarlow/IAD_labs/blob/main/IAD_Lab4P1_Sliepyi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

filename = '/content/drive/MyDrive/Colab Notebooks/ukr.txt'
df = pd.read_csv(filename, sep='\t', header=None, names=['en', 'uk', 'source'], encoding='utf-8', on_bad_lines='skip')
df = df[['en', 'uk']].dropna()

train_df, test_df = train_test_split(df, test_size=0.15, random_state=42)
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'test': Dataset.from_pandas(test_df)
})

model_checkpoint = "Helsinki-NLP/opus-mt-en-uk"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    inputs = examples["en"]
    targets = examples["uk"]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs
tokenized_datasets = dataset.map(preprocess_function, batched=True)

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

args = Seq2SeqTrainingArguments(
    output_dir="my_en_uk_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=False,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Починаємо навчання...")
trainer.train()
trainer.save_model("final_model_en_uk")
print("Модель збережено у папку 'final_model_en_uk'")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.01M [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose "Don't visualize my results"


Epoch,Training Loss,Validation Loss
1,0.529200,0.455190
2,0.412400,0.447452


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[61586]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.529200,0.455190
2,0.412400,0.447452
3,0.344300,0.443902


Модель збережено у папку 'final_model_en_uk'


In [2]:
from transformers import pipeline
import ipywidgets as widgets
from IPython.display import display
model_path = "./final_model_en_uk"
try:
    translator = pipeline("translation", model=model_path, tokenizer=model_path)
    print("Модель успішно завантажена!")
except Exception as e:
    print(f"Помилка завантаження: {e}")
text_input = widgets.Textarea(
    value='This work is not easy.',
    placeholder='Введіть текст англійською',
    description='English:',
    disabled=False
)
button = widgets.Button(description="Перекласти")
output_area = widgets.Output()
def on_button_clicked(b):
    with output_area:
        output_area.clear_output()
        text = text_input.value
        if text:
            result = translator(text)
            print(f"Українська: {result[0]['translation_text']}")
        else:
            print("Введіть текст!")
button.on_click(on_button_clicked)
display(text_input, button, output_area)

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cuda:0


Модель успішно завантажена!


Textarea(value='This work is not easy.', description='English:', placeholder='Введіть текст англійською...')

Button(description='Перекласти', style=ButtonStyle())

Output()

In [5]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 9.1 MB/s eta 0:00:00


In [6]:
from transformers import pipeline
from sacrebleu.metrics import BLEU

translator = pipeline("translation", model="./final_model_en_uk", device=0)
test_sample = test_df.iloc[:50]
refs = [test_sample['uk'].tolist()]
preds = []
for txt in test_sample['en']:
    res = translator(txt, max_length=128)[0]['translation_text']
    preds.append(res)
bleu = BLEU()
score = bleu.corpus_score(preds, refs)
print(f"BLEU Score: {score.score:.2f}")

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cuda:0


Генеруємо переклади для оцінки...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Ваш BLEU Score: 59.10
